In [6]:
import cv2
import mediapipe as mp
import pyautogui

# Initialize MediaPipe and webcam
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)  # You can track one hand
mp_draw = mp.solutions.drawing_utils
cam = cv2.VideoCapture(0)
screen_width, screen_height = pyautogui.size()

while True:
    success, frame = cam.read()
    frame = cv2.flip(frame, 1)
    frame_height, frame_width, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if result.multi_hand_landmarks:
        for hand_landmark in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmark, mp_hands.HAND_CONNECTIONS)

            # Get coordinates of index finger tip (landmark 8)
            index_finger_tip = hand_landmark.landmark[8]
            x = int(index_finger_tip.x * frame_width)
            y = int(index_finger_tip.y * frame_height)

            # Convert to screen coordinates
            screen_x = screen_width * index_finger_tip.x
            screen_y = screen_height * index_finger_tip.y

            # Move the mouse
            pyautogui.moveTo(screen_x, screen_y)

            # Optional: Left click when thumb tip and index tip are close
            thumb_tip = hand_landmark.landmark[4]
            thumb_x = int(thumb_tip.x * frame_width)
            thumb_y = int(thumb_tip.y * frame_height)

            # Draw circle at index finger tip
            cv2.circle(frame, (x, y), 10, (0, 255, 255), -1)

            if abs(x - thumb_x) < 40 and abs(y - thumb_y) < 40:
                pyautogui.click()
                pyautogui.sleep(1)  # Pause to prevent multiple clicks

    cv2.imshow("Virtual Mouse", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC key to exit
        break

cam.release()
cv2.destroyAllWindows()


In [13]:
import cv2
import mediapipe as mp
import pyautogui
import time

# Initialize
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils
cam = cv2.VideoCapture(0)
screen_width, screen_height = pyautogui.size()

# Tap detection
last_click_time = 0
click_interval = 0.4  # Seconds
clicks = 0

prev_scroll_y = None

while True:
    success, frame = cam.read()
    frame = cv2.flip(frame, 1)
    frame_height, frame_width, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if result.multi_hand_landmarks:
        for hand_landmark in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmark, mp_hands.HAND_CONNECTIONS)

            # Finger landmarks
            index_tip = hand_landmark.landmark[8]
            middle_tip = hand_landmark.landmark[12]
            thumb_tip = hand_landmark.landmark[4]

            # Convert to pixel coordinates
            x = int(index_tip.x * frame_width)
            y = int(index_tip.y * frame_height)
            thumb_x = int(thumb_tip.x * frame_width)
            thumb_y = int(thumb_tip.y * frame_height)
            middle_x = int(middle_tip.x * frame_width)
            middle_y = int(middle_tip.y * frame_height)

            # Move mouse
            screen_x = screen_width * index_tip.x
            screen_y = screen_height * index_tip.y
            pyautogui.moveTo(screen_x, screen_y)

            # Draw index circle
            cv2.circle(frame, (x, y), 10, (0, 255, 255), -1)

            # --- Gesture-based Clicks ---

            index_thumb_close = abs(x - thumb_x) < 40 and abs(y - thumb_y) < 40
            middle_thumb_close = abs(middle_x - thumb_x) < 40 and abs(middle_y - thumb_y) < 40

            # ✅ Double Click (Thumb close to both Index and Middle)
            if index_thumb_close and middle_thumb_close:
                print("Double Click Gesture Detected")
                pyautogui.doubleClick()
                time.sleep(1)

            # ✅ Single Left Click (Only Thumb close to Index)
            elif index_thumb_close:
                print("Single Click Gesture Detected")
                pyautogui.click()
                time.sleep(1)

            # ✅ Right Click (Only Thumb close to Middle)
            elif middle_thumb_close:
                print("Right Click Gesture Detected")
                pyautogui.rightClick()
                time.sleep(1)


            # --- Scroll (Index & Middle move vertically together) ---
            if abs(index_x := x - middle_x) < 40:  # fingers close horizontally
                avg_y = (index_tip.y + middle_tip.y) / 2
                scroll_y = int(avg_y * frame_height)

                if prev_scroll_y is not None:
                    dy = scroll_y - prev_scroll_y
                    if abs(dy) > 5:  # more sensitive to small movement
                        scroll_speed = int(dy * 3)  # amplify movement (adjust multiplier as needed)
                        pyautogui.scroll(-scroll_speed)  # negative = down, positive = up


                prev_scroll_y = scroll_y
            else:
                prev_scroll_y = None  # reset if fingers not close

    cv2.imshow("Virtual Mouse", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cam.release()
cv2.destroyAllWindows()


Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Double Click Gesture Detected
Double Click Gesture Detected
Double Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Right Click Gesture Detected
Right Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Single Click Gesture Detected
Right Click Gesture Detected
